# 07 · Sensitivity runs (S1)
Two extra arms of the per-class chronological protocol, at the selected window length (L = 8), multi-class task, ten seeds:

* **SC_raw**: chronological split on the **raw** data (no deduplication). Completes the 2 × 2 design (random/chronological × raw/deduplicated), so the effect of deduplication can be separated from the effect of the split. Runs on both datasets.
* **SC_feat**: chronological split after **label-free** deduplication (identical feature vectors removed regardless of label; the earliest copy is kept). Labels never decide which rows enter the test set. Only CICIDS2017 is run, because InSDN has no conflicting labels.

Estimated time on the laptop: roughly 10–14 h for SC_raw (both datasets) and 6–8 h for SC_feat. Resumable as before: if it stops, run it again.

In [ ]:
import sys, os, json, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
pd.set_option('display.width', 220, 'display.max_columns', 60, 'display.max_rows', 200)
from sdnids_bench.config import load_config, mode_params, active_datasets, enable_determinism
cfg = load_config('..')
mp = mode_params(cfg)
print('mode:', cfg['mode'], '| results:', cfg['out_dir'])
enable_determinism()
from sdnids_bench.experiments import run_grid, load_results
L = 8
SEEDS = mp['seeds']
print('L =', L, '| seeds:', SEEDS)

In [ ]:
for ds in active_datasets(cfg, 'B1'):
    run_grid(cfg, 'S1', [ds], ['multi'], ['SC_raw'], [L], SEEDS, cfg['models'])

In [ ]:
if 'CICIDS2017' in active_datasets(cfg, 'B1'):
    run_grid(cfg, 'S1', ['CICIDS2017'], ['multi'], ['SC_feat'], [L], SEEDS, cfg['models'])

In [ ]:
res = load_results(cfg, active_datasets(cfg, 'B1'))
cmp = res[((res.exp == 'S1') | ((res.exp == 'B1') & (res.protocol.isin(['SC', 'SR_raw', 'SR_dedup'])))) &
          (res.task == 'multi') & (res.L == L)]
tab = cmp.pivot_table(index=['dataset', 'model'], columns='protocol', values='macro_f1', aggfunc='mean').round(4)
display(tab)
os.makedirs(os.path.join(cfg['out_dir'], 'paper'), exist_ok=True)
tab.to_csv(os.path.join(cfg['out_dir'], 'paper', 'sensitivity_macro_f1.csv'))